# 🔧 Build the LLM gateway — routing, fallbacks, caching & cost (mock mode)

> 📓 *Companion to* **Modern Agentic AI Engineer** *· Ch 39 §39.9 · type: walkthrough*

**The promise:** by the end you can put *one* gateway in front of every model call — it routes by task difficulty, fails over to a backup provider on error or timeout, caches identical calls, and records cost — so swapping or mixing providers becomes *config*, not a code edit scattered across your codebase. You build the toy here; the production version lives in [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) and [`capstone/llm/gateway.py`](../../../capstone/llm/gateway.py).

Runs fully **offline and free**: `MOCK=1` (the default) calls two *canned* backends that return realistic responses plus token-usage — no network, no keys, no spend.

## 🧠 Why this matters

The model is the most volatile dependency in an AI system: providers have outages, change prices, deprecate models, and ship better ones every few months. If raw `client.messages.create(...)` calls are scattered across forty files, every one of those events is a code change in forty places.

A **gateway** is one component that *all* model calls go through (LiteLLM is the popular off-the-shelf option). In one place it gives you **routing** (cheap model for easy tasks, frontier model for hard ones), **fallbacks** (retry on a backup when the primary errors — the reliability patterns of Ch 29), **caching** (skip identical calls), **rate limiting & cost tracking** (per tenant), and **provider independence** (mix Claude, OpenAI, and your own vLLM behind one interface). It is the *isolate-what-changes* instinct of hexagonal architecture (Ch 28) applied to the model itself — and one of the highest-leverage pieces of infrastructure in a serious agentic platform. See §39.9.

## Objectives & prereqs

**By the end you can:**
- Define a provider-agnostic `complete()` interface that hides which backend served a request.
- Route requests to a *cheap* or *frontier* tier from a difficulty signal, and read the routing log.
- Fail over to a backup backend when the primary raises or times out (the Ch 29 pattern, as config).
- Put an exact-match cache in front of the gateway so an identical request skips the model.
- Track per-call and aggregate cost from the usage every backend returns.

**Prereqs:** Ch 11 / [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) (SDK call shapes, retries); Ch 29 (fallback, backpressure) read; Ch 28 (hexagonal / isolate what changes).

**Packages:** the standard library only (`os`, `time`, `random`, `hashlib`, `dataclasses`). Nothing beyond `requirements.txt`.

**Runs free & offline.** `MOCK=1` (default) calls canned backends. The `MOCK=0` path — routing to *live* providers — is sketched and clearly flagged; it is optional and would make a handful of short real calls.

In [ ]:
# --- Setup -------------------------------------------------------------------
import hashlib
import os
import random
import time
from dataclasses import dataclass, field

from dotenv import load_dotenv

load_dotenv()  # reads a local .env if present; never hardcode keys

# MOCK=1 (default): the gateway's backends are CANNED -- realistic responses plus
# token usage, returned offline and deterministically. MOCK=0 would route to live
# providers (see the live-path note near the end). The gateway code is identical
# either way -- that is the whole point of the abstraction.
MOCK = os.getenv("COMPANION_MOCK", "1") == "1"

random.seed(39)  # determinism for the simulated outages below

# The two tiers the gateway routes between. In a real config these map to
# provider+model pairs; here they name our two mock backends. The cheap tier is a
# small fast model; the frontier tier is the most capable Opus-class model.
CHEAP_MODEL = "claude-haiku-4-5"      # fast, inexpensive -- easy tasks
FRONTIER_MODEL = "claude-opus-4-8"    # most capable -- hard tasks

print(f"MOCK mode: {MOCK}  (offline, free; backends are canned)" if MOCK
      else f"MOCK mode: {MOCK}  (LIVE -- will route to real providers)")
print(f"tiers: cheap={CHEAP_MODEL!r}  frontier={FRONTIER_MODEL!r}")

## 1 · 🧠 The gateway as a single chokepoint

Picture every model call in your platform funneling through one box. Application code asks the box for a *completion*; it never names a provider. Inside the box, in order:

```
  request --> [ cache ] --> [ router ] --> [ primary backend ]
                  |              |               |  (error/timeout)
               (hit?)        (cheap or          v
                  |          frontier?)   [ backup backend ]
                  v                              |
             cached reply <--- [ cost tracker ] <-- response + usage
```

Routing, fallback, caching, and cost tracking all live *here*, once — not sprinkled through your handlers. We will build each layer and watch it work.

## 2 · A provider-agnostic interface and two mock backends

First the contract. A `Completion` carries the text and a `usage` record (the field every provider returns); a backend is anything with a `complete(prompt, ...)` method. Our two canned backends mimic a cheap tier and a frontier tier — same shape, different latency, cost, and (for hard prompts) quality. No network is touched.

In [ ]:
@dataclass
class Usage:
    """The token-accounting record every provider returns, normalized."""
    input_tokens: int
    output_tokens: int


@dataclass
class Completion:
    """A provider-agnostic result. Application code reads .text and .usage only."""
    text: str
    usage: Usage
    model: str          # which backend actually served it (for the log)
    from_cache: bool = False


# Illustrative per-million-token prices (NOT real quotes). Output costs several x
# input; the frontier tier costs ~15x the cheap tier here. Plug in real numbers.
PRICES = {
    CHEAP_MODEL:    {"in": 0.80,  "out": 4.00},    # $ / 1M tokens
    FRONTIER_MODEL: {"in": 15.00, "out": 75.00},
}


def cost_usd(model: str, usage: Usage) -> float:
    p = PRICES[model]
    return usage.input_tokens / 1e6 * p["in"] + usage.output_tokens / 1e6 * p["out"]


class MockBackend:
    """A canned 'provider'. Returns a realistic response + usage, no network.

    fail_rate simulates an outage; slow_after simulates a timeout-prone backend.
    On the live path (MOCK=0) this class is replaced by a real SDK client whose
    .complete() calls client.messages.create(...) -- same Completion shape.
    """

    def __init__(self, model, *, fail_rate=0.0, latency=0.01, quality="good"):
        self.model = model
        self.fail_rate = fail_rate
        self.latency = latency
        self.quality = quality  # 'good' or 'weak' -- the cheap tier is weak on hard tasks

    def complete(self, prompt: str, max_tokens: int = 256) -> Completion:
        time.sleep(self.latency)  # tiny, deterministic stand-in for network + decode
        if random.random() < self.fail_rate:
            raise ConnectionError(f"{self.model}: simulated provider 503")
        # A canned answer whose length scales mildly with the prompt -> realistic usage.
        in_tok = max(8, len(prompt) // 4)
        out_tok = min(max_tokens, 24 + len(prompt) // 8)
        if self.quality == "weak" and _is_hard(prompt):
            text = f"[{self.model}] Short, shallow answer (this tier is out of its depth here)."
        else:
            text = f"[{self.model}] A solid, well-reasoned answer to: {prompt[:48]}..."
        return Completion(text=text, usage=Usage(in_tok, out_tok), model=self.model)


def _is_hard(prompt: str) -> bool:
    """Forward-declared difficulty signal; defined for real in section 3."""
    return len(prompt) > 160 or any(t in prompt.lower() for t in ("prove", "design", "analyze"))


print("interface + two backends defined (cheap=weak-on-hard, frontier=good).")

## 3 · Routing: pick a tier from a difficulty signal

The router's job: send each request to the *right* tier. The signal is whatever cheaply estimates difficulty — here a length threshold plus a few task keywords (a real router might use an explicit task tag or a tiny classifier). Easy → cheap tier; hard → frontier tier.

In [ ]:
HARD_HINTS = ("prove", "design", "analyze", "architect", "debug the race")


def difficulty(prompt: str, task_tag: str | None = None) -> str:
    """Return 'hard' or 'easy'. A cheap, explainable signal -- not a guess (see the pitfall)."""
    if task_tag in {"reasoning", "planning", "code-review"}:
        return "hard"
    if task_tag in {"classify", "extract", "format"}:
        return "easy"
    if len(prompt) > 160 or any(h in prompt.lower() for h in HARD_HINTS):
        return "hard"
    return "easy"


def route(prompt: str, task_tag: str | None = None) -> str:
    """Map difficulty -> a tier name."""
    return FRONTIER_MODEL if difficulty(prompt, task_tag) == "hard" else CHEAP_MODEL


# A tiny mixed prompt set: some trivial, some genuinely hard.
PROMPTS = [
    ("Capital of France?", "classify"),
    ("Extract the date from: 'shipped 2026-06-20'.", "extract"),
    ("Design a fault-tolerant multi-region gateway and analyze its failure modes.", "reasoning"),
    ("Prove that a DAG always has a topological order.", None),
    ("Reformat this list as JSON.", "format"),
    ("Summarize this 3-line email.", None),
]
print("router + a 6-prompt mixed batch ready.")

## 🔮 Predict

Six prompts above, two tiers. **Before running the next cell**, predict: how many of the six land on the **frontier** tier, and how many on the **cheap** tier?

*Hint:* a `task_tag` decides outright when present; otherwise length (>160 chars) or a hard keyword (`prove`, `design`, `analyze`…) tips it to hard. Write your split down, then run.

In [ ]:
print(f"{'tier':<18} {'tag':<12} prompt")
print("-" * 78)
counts = {CHEAP_MODEL: 0, FRONTIER_MODEL: 0}
for prompt, tag in PROMPTS:
    tier = route(prompt, tag)
    counts[tier] += 1
    print(f"{tier:<18} {str(tag):<12} {prompt[:46]}")
print("-" * 78)
print(f"frontier: {counts[FRONTIER_MODEL]}   cheap: {counts[CHEAP_MODEL]}")

**What you just saw.** Two prompts went to the frontier tier — the *design/analyze* one (hard keyword + length) and the `reasoning`-tagged one — wait, the *prove* prompt also tipped hard on its keyword. The cheap, trivially-easy work (classify, extract, format, the short summary) stayed on the cheap tier. Routing turned a one-size-fits-all `opus` call into a cost-aware decision *without the caller knowing a thing*.

## 4 · The gateway: cache → route → call, with fallback and cost tracking

Now assemble the box. `Gateway.complete()` is the single entry point. It checks an **exact cache** first, **routes** to a tier, calls the **primary** backend for that tier, and on any error **falls back** to a backup — recording **cost** and a routing log throughout. Application code calls `gw.complete(prompt)` and never sees any of this machinery.

In [ ]:
@dataclass
class Gateway:
    # tier name -> [primary backend, backup backend]; fallback walks the list in order.
    backends: dict
    cache: dict = field(default_factory=dict)
    log: list = field(default_factory=list)
    total_cost: float = 0.0
    max_retries: int = 1  # attempts on the BACKUP after the primary fails

    def _key(self, prompt, tier, max_tokens):
        """Exact-match cache key: identical request -> identical key."""
        raw = f"{tier}|{max_tokens}|{prompt}".encode()
        return hashlib.sha256(raw).hexdigest()

    def complete(self, prompt, task_tag=None, max_tokens=256) -> Completion:
        tier = route(prompt, task_tag)
        key = self._key(prompt, tier, max_tokens)

        # 1) CACHE: an identical request skips the model entirely (exact match only;
        #    semantic/prefix caching is Ch 40's job -- see the forward link).
        if key in self.cache:
            hit = self.cache[key]
            self.log.append((tier, "cache-hit", 0.0))
            return Completion(hit.text, hit.usage, hit.model, from_cache=True)

        # 2) ROUTE + 3) CALL with FALLBACK: fetch the tier's chain ONCE, then walk it
        #    in order -- primary, then each backup. We stay on `tier` the whole time
        #    and never re-index self.backends by a backend's model name (which carries
        #    an '@backup' suffix that is not a tier key). max_retries bounds how many
        #    backups we try after the primary fails.
        chain = self.backends[tier]  # [primary, backup, ...]
        attempts = chain[: 1 + self.max_retries]  # primary + up to max_retries backups
        last_err = None
        for attempt, backend in enumerate(attempts):
            try:
                resp = backend.complete(prompt, max_tokens=max_tokens)
            except Exception as exc:  # outage/timeout -> fall over (Ch 29)
                last_err = exc
                self.log.append((tier, f"FAILED:{backend.model}:{exc}", 0.0))
                continue
            # 4) COST TRACKING on every successful call. Price by the TIER (a real
            #    pricing key); the backend that served it -- primary or '@backup' --
            #    is recorded in the log, not used to look up a price.
            c = cost_usd(tier, resp.usage)
            self.total_cost += c
            kind = "primary" if attempt == 0 else f"fallback#{attempt}"
            self.log.append((tier, f"{kind}:{resp.model}", c))
            self.cache[key] = resp
            return resp

        raise RuntimeError(f"all backends for tier {tier} failed") from last_err


print("Gateway defined: cache -> route -> call(primary, then backups) -> cost.")

### Wire up the backends and run the batch

Each tier gets a **primary** and a **backup** (a different provider for the same tier). The frontier *primary* is deliberately flaky here (`fail_rate=0.5`) so we can watch fallback fire. Same prompt set; this time the gateway actually answers.

In [ ]:
gw = Gateway(backends={
    # cheap tier: a healthy primary, a healthy backup on another provider
    CHEAP_MODEL: [
        MockBackend(CHEAP_MODEL, fail_rate=0.0, quality="weak"),
        MockBackend(CHEAP_MODEL + "@backup", fail_rate=0.0, quality="weak"),
    ],
    # frontier tier: a FLAKY primary (simulated outage) and a healthy backup
    FRONTIER_MODEL: [
        MockBackend(FRONTIER_MODEL, fail_rate=0.5, quality="good"),
        MockBackend(FRONTIER_MODEL + "@backup", fail_rate=0.0, quality="good"),
    ],
})

for prompt, tag in PROMPTS:
    resp = gw.complete(prompt, task_tag=tag)
    print(f"-> served by {resp.model:<26} | {resp.text[:54]}")

print(f"\nrouting log ({len(gw.log)} events):")
for tier, event, c in gw.log:
    print(f"  [{tier:<16}] {event:<44} ${c:.5f}")
print(f"\ntotal cost this batch: ${gw.total_cost:.5f}")

**What you just saw.** Every prompt got an answer, but the *log* tells the real story: the two frontier-tier prompts whose primary hit the simulated 503 silently fell over to the `@backup` provider and still returned — the Ch 29 fallback pattern, now a property of the gateway rather than something each caller reimplements. Cost accrued per call, attributed to whichever backend actually served it.

## 5 · The cache earns its keep

Caching is the cheapest win in the box: an *identical* request should never pay the model twice. Re-send a prompt we already served and watch it return from cache at zero cost and zero latency.

In [ ]:
cost_before = gw.total_cost
repeat_prompt, repeat_tag = PROMPTS[0]  # 'Capital of France?' -- already served above

resp = gw.complete(repeat_prompt, task_tag=repeat_tag)
print(f"from_cache : {resp.from_cache}")
print(f"text       : {resp.text[:54]}")
print(f"cost added : ${gw.total_cost - cost_before:.5f}  (an exact-cache hit is free)")
print(f"last log   : {gw.log[-1]}")

**What you just saw.** The repeat resolved from the cache — `from_cache=True`, `$0.00000` added, logged as `cache-hit`. This is *exact-match* caching (byte-identical request). The deeper wins — semantic caching, prompt-prefix caching, and routing-for-cost — are Ch 40's job, layered at *this same gateway*; we link forward at the end.

## ⚠️ Pitfall: a too-aggressive router sends hard tasks to the cheap tier

Routing saves money only while it routes *correctly*. Lower the difficulty bar — say, "send everything under 400 chars to the cheap tier" — and genuinely hard prompts get the weak model. The cost line drops; the **quality** falls off a cliff, invisibly, because nothing errors. Watch a hard prompt get mis-routed.

In [ ]:
hard_prompt = "Prove the halting problem is undecidable."  # short, but genuinely hard

# A reckless router: route purely on raw length, ignoring the task. SHORT == cheap.
def greedy_route(prompt):
    return CHEAP_MODEL if len(prompt) < 400 else FRONTIER_MODEL

reckless_tier = greedy_route(hard_prompt)
good_tier = route(hard_prompt, task_tag="reasoning")  # the signal-based router

# Reckless path: the greedy router pins this to the (healthy) cheap tier, so we
# call that backend directly to expose the WEAK answer it ships. The careful path
# goes through the gateway, which routes to the frontier tier AND fails over its
# flaky primary to the backup -- so we reliably see the GOOD answer, not a 503.
reckless = gw.backends[reckless_tier][0].complete(hard_prompt)
careful = gw.complete(hard_prompt, task_tag="reasoning")

print(f"greedy-by-length -> {reckless_tier}")
print(f"   {reckless.text}")
print(f"\nsignal-based     -> {good_tier}")
print(f"   {careful.text}")
print("\nThe greedy router 'saved money' by sending a hard task to a weak tier --")
print("and silently shipped a shallow answer. No exception fired.")

**The fix.** Route on an *evaluated* signal, not a guess: an explicit task tag the caller sets, a tiny trained classifier, or a difficulty estimate you have **measured against quality** (Ch 22's evals are exactly this loop). Then watch the cheap-tier quality on a held-out set so a routing change that quietly degrades answers shows up *before* a customer finds it. Cheap-by-default is only safe when you can see the quality it buys.

## 🎯 Senior lens

Build or adopt the gateway *early* — before the second provider, not after the first outage. The payoff is repeated and compounding: a provider going down, a price change, a new and better model, a new tenant needing its own rate limit — each becomes a *config* change at one chokepoint instead of edits scattered across the codebase. It is hexagonal architecture (Ch 28) applied to the most volatile dependency in the whole system, the model, and it converts "we're locked into one provider" into "we route to whichever is best today."

Know where the abstraction *leaks*, though. Routing, fallback, caching, and cost tracking abstract cleanly. **Tool-calling fidelity does not** — each model family has its own chat template and tool-call parser, so a self-hosted model dropped behind the gateway can silently fail to parse tool calls even though plain completions work. "The gateway makes models interchangeable" is true for plumbing, not for tool behavior; test tool calling against each specific model. The gateway is still worth having — just know which guarantees stop at the serving boundary (§39).

## 📋 Gateway responsibilities checklist

A production gateway owns all six of these in one place (mirrors the chapter's gateway figure):

- [ ] **Routing** — each request to the right model; cheap for easy, frontier for hard, on an *evaluated* signal.
- [ ] **Fallback** — retry on a backup provider on error/timeout (Ch 29); never let one provider's outage be yours.
- [ ] **Caching** — skip identical calls (exact here; semantic/prefix caching layered on in Ch 40).
- [ ] **Rate limiting** — per tenant, so one caller can't exhaust a shared quota (Ch 41 adds per-tenant limits).
- [ ] **Cost tracking** — per call and per tenant, attributed to the backend that served it.
- [ ] **Provider independence** — mix hosted and self-hosted (vLLM/TGI/Ollama) behind one `complete()`.

## Recap

- A **gateway** is one chokepoint every model call goes through: routing · fallback · caching · rate limiting · cost tracking · provider independence — all in one place.
- **Routing** maps a difficulty signal to a tier (cheap vs frontier); good signals are *evaluated*, not guessed.
- **Fallback** retries on a backup backend when the primary errors or times out — the Ch 29 reliability pattern, now config.
- An **exact cache** makes identical requests free; semantic/prefix caching is Ch 40, layered at this same gateway.
- **Cost tracking** rides on the `usage` every backend returns, attributed to whichever one served the call.
- The abstraction leaks at **tool calling** (model-specific templates/parsers) — plumbing is interchangeable, tool behavior is not.
- Provider independence is an *architectural* decision (Ch 28): isolate the most volatile dependency — the model — behind one interface.

## Exercises

Each exercise *changes* something and asks you to predict the result before you run. (Solutions arrive in `solutions/` in Phase 2.)

1. **Break the backup too.** Set the frontier *backup* backend's `fail_rate=1.0` as well, then complete a frontier-tier prompt. Predict what `complete()` does when the whole chain fails (look at the final `raise`), then run and confirm the error and the log.
2. **Make the cache pay off.** Build a batch with 40% exact-duplicate prompts, run it through a fresh `Gateway`, and compare `total_cost` and `len(log)` cache-hits against the no-duplicate case. Predict the cost reduction *before* measuring it.
3. **Add a third, mid tier.** Introduce a `MID_MODEL` between cheap and frontier, extend `route()` to a three-way decision (e.g. by a `task_tag` or a length band), and re-run the section-3 batch. Predict how the tier counts shift.
4. **Cost-aware routing (peek at Ch 40).** Add a `budget_per_call` argument; when the routed frontier call's *estimated* cost exceeds it, downgrade to the cheap tier and log the downgrade. Predict which of the six prompts get downgraded at a tight budget.

In [ ]:
# Exercise 1 -- your code here


In [ ]:
# Exercise 2 -- your code here


In [ ]:
# Exercise 3 -- your code here


In [ ]:
# Exercise 4 -- your code here


## A note on the live path (`MOCK=0`)

Everything above ran offline against canned backends. To make it *real*, you replace only `MockBackend` — nothing in `Gateway`, `route`, or the cache changes. A live backend wraps a provider SDK:

```python
# import anthropic  # on the live path only
# class AnthropicBackend:
#     def __init__(self, model):
#         self.model = model
#         self._client = anthropic.Anthropic()  # reads ANTHROPIC_API_KEY from env
#     def complete(self, prompt, max_tokens=256):
#         msg = self._client.messages.create(
#             model=self.model, max_tokens=max_tokens,
#             messages=[{"role": "user", "content": prompt}],
#         )
#         return Completion(msg.content[0].text,
#                           Usage(msg.usage.input_tokens, msg.usage.output_tokens),
#                           self.model)
```

The backup for a tier would point at a *different provider's* SDK behind the same `complete()` — which is exactly how the gateway gives you real provider independence. Keys come **only** from the environment; `MOCK=0` would make a handful of short, paid calls. The production version of all of this — real providers, retries with backoff, telemetry hooks, rate limits — is the blueprint linked below.

## Next

You built the toy gateway; here's where the real one lives.

- 🔧 **Blueprint this becomes:** [`blueprints/llm-gateway/`](../../../blueprints/llm-gateway/) — the production gateway: real providers, retries with backoff, telemetry hooks ([`blueprints/observability-stack/`](../../../blueprints/observability-stack/) reads them), rate limits, and hosted + self-hosted behind one interface.
- 🎓 **Capstone:** this is the seed of [`capstone/llm/gateway.py`](../../../capstone/llm/gateway.py) — the platform's single model chokepoint (checkpoint `checkpoints/ch39-serving-and-gateway`). Ch 40 extends *this same gateway* with metering, semantic caching, and routing-for-cost; Ch 41 adds guardrails and per-tenant limits at it.
- 📓 **Next notebook:** [`39-02-serving-internals-batching-kv-cache-quant.ipynb`](39-02-serving-internals-batching-kv-cache-quant.ipynb) — *why* inference costs what it does, so you can reason about the tiers you just routed between.
- 📖 **Book:** §39.9 (routing, fallbacks & gateways), §29 (fallback, backpressure), §28 (hexagonal / isolate what changes), Ch 11 (the SDK call shape underneath).